In [3]:
import pandas as pd

In [4]:
# df=pd.read_csv("D:\Resume Project DS 1\data\raw\nfhs5_district_kaggle.csv")  ---> GAVE ERROR
# df = pd.read_csv("D:\\Resume Project DS 1\\data\\raw\\nfhs5_district_kaggle.csv") --WORKS  
# df = pd.read_csv(r"D:\Resume Project DS 1\data\raw\nfhs5_district_kaggle.csv") -->Use raw strings: Prepend the string with r to tell Python to treat backslashes as literal characters
df = pd.read_csv('../data/raw/nfhs5_district_kaggle.csv')

In [5]:
# Sanity check
print(df.shape)

(706, 109)


In [6]:
# Missing value count + percentage
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df)) * 100
})

In [7]:
# Show only columns that HAVE missing values
missing = missing[missing['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missing)

Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []


In [8]:
# Check what suspicious values exist
for col in df.columns:
    unique_vals = df[col].unique()
    for val in unique_vals:
        if str(val).strip() in ['-', 'N/A', 'NA', '..', '', 'n/a', 'null']:
            print(f"{col}: '{val}'")

In [14]:
# Look at unique values in target columns
# print(df['Children under 5 years who are stunted (height-for-age)_Total'].unique()[:20])
# print(df['Children under 5 years who are wasted (weight-for-height)_Total'].unique()[:20])
# print(df['Children under 5 years who are underweight (weight-for-age)_Total'].unique()[:20])

In [15]:
# Find target column names
for col in df.columns:
    if 'stunt' in col.lower() or 'wast' in col.lower() or 'underweight' in col.lower():
        print(col)

Children under 5 years who are stunted (height-for-age)18 (%)
Children under 5 years who are wasted (weight-for-height)18 (%)
Children under 5 years who are severely wasted (weight-for-height)19 (%)
Children under 5 years who are underweight (weight-for-age)18 (%)


In [16]:
stunt_col = 'Children under 5 years who are stunted (height-for-age)18 (%)'
wast_col = 'Children under 5 years who are wasted (weight-for-height)18 (%)'
under_col = 'Children under 5 years who are underweight (weight-for-age)18 (%)'

print(df[stunt_col].unique()[:20])
print(df[wast_col].unique()[:20])
print(df[under_col].unique()[:20])

['21.6 ' '27.0 ' '21.1 ' '19.7 ' '36.4 ' '31.0 ' '23.1 ' '31.4 ' '29.8 '
 '23.8 ' '22.6 ' '29.2 ' '34.4 ' '50.5 ' '36.0 ' '27.1 ' '30.4 ' '24.2 '
 '35.7 ' '29.7 ']
['15.7 ' '27.0 ' '12.6 ' '19.5 ' '19.2 ' '21.5 ' '14.3 ' '11.7 ' '17.8 '
 '8.7 ' '17.2 ' '14.1 ' '16.7 ' '19.3 ' '14.8 ' '7.1 ' '23.2 ' '15.8 '
 '9.0 ' '12.2 ']
['24.6 ' '42.8 ' '17.4 ' '21.4 ' '32.2 ' '33.5 ' '22.4 ' '22.5 ' '21.1 '
 '26.9 ' '24.7 ' '27.8 ' '26.7 ' '46.3 ' '40.6 ' '27.9 ' '9.0 ' '13.4 '
 '14.2 ' '15.6 ']


In [ ]:
# Strip trailing/leading spaces from all object columns
df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Now convert target columns to float
df[stunt_col] = pd.to_numeric(df[stunt_col], errors='coerce')
df[wast_col] = pd.to_numeric(df[wast_col], errors='coerce')
df[under_col] = pd.to_numeric(df[under_col], errors='coerce')

# Verify
print(df[stunt_col].dtype)
print(df[stunt_col].isnull().sum())  # check if any became NaN after conversion

float64
1


In [18]:
# print(df[df[stunt_col].isnull()][['State', 'District', stunt_col]])

In [19]:
# Find actual column names
print(df.columns[:5].tolist())

['District Names', 'State/UT', 'Number of Households surveyed', 'Number of Women age 15-49 years interviewed', 'Number of Men age 15-54 years interviewed']


In [20]:
print(df[df[stunt_col].isnull()][['State/UT', 'District Names', stunt_col]])

           State/UT District Names  \
330  Madhya Pradesh       Jabalpur   

     Children under 5 years who are stunted (height-for-age)18 (%)  
330                                                NaN              


## Day 7 finding — note this in a markdown cell
### Jabalpur (MP) has 1 missing stunting value after conversion
### Action: Fill with median in Day 8 during imputation step

Day 7 is now complete. Here's what you found today:

✅ No real NaN values — hidden as trailing spaces
✅ Fixed all object columns with str.strip()
✅ Target columns converted to float64
✅ 1 genuine missing value — Jabalpur, MP (stunting only)